# Phase A (Notebook 3): Single-Pass `deepseek-flash` Vision API Scene Captioning (`PhaseA3_VLM_Captioning.ipynb`)

**Environment Stack**: Lightweight `deepseek-flash` Vision API client (`openai>=1.50.0`, `pillow`) running on a **free Colab CPU runtime** (0 GPU compute units required).

### Workflow Overview
1. **Cell 1 — Google Drive Mount, API Key Check & Input Verification**:
   - Mounts Drive, loads `DEEPSEEK_API_KEY` from **Colab Secrets** (key icon on left sidebar) or optional cell input, and checks `shots_and_scenes.json`, `attributed_subtitles.json`, and `keyframes/`.
2. **Cell 2 — Clone / Pull Latest Code from GitHub (`/content/MovieRag_repo`)**:
   - Clones/pulls `main`, including `char_bank/sintel/external_plot.txt` and `pipeline/stage5_vlm_caption.py`.
3. **Cell 3 — Install Lightweight API Dependencies**:
   - Installs `openai>=1.50.0` and `pillow` in `< 5` seconds.
4. **Cell 4 — Review & Apply Optional Manual Subtitle Speaker Overrides**:
   - Applies `SPEAKER_OVERRIDES` (if any) and prints the exact dialogue lines going into the VLM prompt.
5. **Cell 5 (Optional 1-Scene Diagnostic) — Test Scene 4 (`sintel_scene_0004`)**:
   - Sends Scene 4 (up to 15 keyframes downscaled to `960px` width in memory) to `deepseek-flash` (`thinking: disabled`) and prints `vlm_caption`, `exit_state`, and token usage.
6. **Cell 6 — Stage A5 (`stage5_vlm_caption.py`) Full 34-Scene Run & Bundle Assembly**:
   - Archives any old `vlm_captions.jsonl` / `movie_knowledge_bundle.json` into `archive_prev_vlm/` when `RESTART_STAGE5 = True`, captions all 34 scenes in single-pass mode, and writes `movie_knowledge_bundle.json`.
7. **Cell 7 — Knowledge Bundle Verification & Inspection**:
   - Validates schema completeness, `external_plot_documents`, subtitles, `vlm_caption`, and `exit_state` across sample scenes.

In [ ]:
# ==============================================================================
# CELL 1: Google Drive Mount, DeepSeek API Key & Stage A1-A4 Artifact Check
# ==============================================================================
import os

try:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not running inside Google Colab; using local paths.")

MOVIE_ID = "sintel"                              # @param {type:"string"}
MOVIE_TITLE = "Sintel"                           # @param {type:"string"}
FRANCHISE_ID = "sintel"                          # @param {type:"string"}
DRIVE_ROOT = "/content/drive/MyDrive/MovieRAG"   # @param {type:"string"}
VLM_MODEL_ID = "deepseek-flash"                  # @param {type:"string"}
API_BASE_URL = "https://api.deepseek.com"        # @param {type:"string"}
MAX_KEYFRAMES_PER_SCENE = 15                     # @param {type:"integer"}
MAX_IMAGE_WIDTH = 960                            # @param {type:"integer"}
DEEPSEEK_API_KEY_OVERRIDE = ""                   # @param {type:"string"}

CHAR_BANK_DIR = f"{DRIVE_ROOT}/char_bank/{MOVIE_ID}"
WORKDIR = f"{DRIVE_ROOT}/workdir/{MOVIE_ID}"

# Load DEEPSEEK_API_KEY from override field, Colab Secrets, or environment
_api_key = DEEPSEEK_API_KEY_OVERRIDE.strip() or os.environ.get("DEEPSEEK_API_KEY", "").strip()
if not _api_key and IN_COLAB:
    try:
        _api_key = (userdata.get("DEEPSEEK_API_KEY") or "").strip()
    except Exception:
        _api_key = ""

if _api_key:
    os.environ["DEEPSEEK_API_KEY"] = _api_key
    print(f"[OK] DEEPSEEK_API_KEY loaded (starts with '{_api_key[:6]}...')")
else:
    print("[WARNING] DEEPSEEK_API_KEY not found yet! Add 'DEEPSEEK_API_KEY' in Colab Secrets (key icon on left sidebar) or paste it into DEEPSEEK_API_KEY_OVERRIDE above.")

for label, path in [
    ("shots_and_scenes.json", f"{WORKDIR}/shots_and_scenes.json"),
    ("attributed_subtitles.json", f"{WORKDIR}/attributed_subtitles.json"),
    ("keyframes dir", f"{WORKDIR}/keyframes"),
    ("character_glossary.json", f"{CHAR_BANK_DIR}/character_glossary.json"),
]:
    ok = os.path.exists(path)
    print(f"  [{'OK' if ok else 'MISSING'}] {label}: {path}")

In [ ]:
# ==============================================================================
# CELL 2: Clone / Pull Latest Source Code from GitHub (/content/MovieRag_repo)
# ==============================================================================
import importlib
import os
import subprocess
import sys

GITHUB_REPO_URL = "https://github.com/ctu-flix/MovieRag.git"  # @param {type:"string"}
GITHUB_BRANCH = "main"                                        # @param {type:"string"}
REPO_SUBDIR = ""
CLONE_DIR = "/content/MovieRag_repo"

github_token = None
if globals().get("IN_COLAB", False):
    try:
        from google.colab import userdata
        github_token = userdata.get("GITHUB_TOKEN")
    except Exception:
        github_token = None

auth_repo_url = GITHUB_REPO_URL
if github_token and GITHUB_REPO_URL.startswith("https://github.com/"):
    auth_repo_url = GITHUB_REPO_URL.replace("https://github.com/", f"https://{github_token}@github.com/")

if not os.path.isdir(os.path.join(CLONE_DIR, ".git")):
    print(f"Cloning {GITHUB_REPO_URL} (branch: {GITHUB_BRANCH})...")
    subprocess.run(["git", "clone", "--branch", GITHUB_BRANCH, auth_repo_url, CLONE_DIR], check=True)
else:
    print(f"Pulling latest commits from origin/{GITHUB_BRANCH}...")
    subprocess.run(["git", "-C", CLONE_DIR, "remote", "set-url", "origin", auth_repo_url], check=True)
    subprocess.run(["git", "-C", CLONE_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", CLONE_DIR, "checkout", GITHUB_BRANCH], check=True)
    subprocess.run(["git", "-C", CLONE_DIR, "pull", "--ff-only", "origin", GITHUB_BRANCH], check=True)

if REPO_SUBDIR and os.path.isdir(os.path.join(CLONE_DIR, REPO_SUBDIR)):
    PROJECT_ROOT = os.path.join(CLONE_DIR, REPO_SUBDIR)
elif os.path.isdir(os.path.join(CLONE_DIR, "MovieRag", "pipeline")):
    PROJECT_ROOT = os.path.join(CLONE_DIR, "MovieRag")
else:
    PROJECT_ROOT = CLONE_DIR

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

for mod_name in [k for k in list(sys.modules) if k == "pipeline" or k.startswith("pipeline.")]:
    importlib.reload(sys.modules[mod_name])

commit_log = subprocess.run(["git", "-C", CLONE_DIR, "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip()
print(f"Active PROJECT_ROOT: {PROJECT_ROOT}")
print(f"Latest Commit      : {commit_log}")

In [ ]:
# ==============================================================================
# CELL 3: Install Lightweight OpenAI / DeepSeek Vision Client Dependencies
# ==============================================================================
import subprocess
import sys

pkgs = [
    "openai>=1.50.0",
    "pillow",
]
print("Installing API dependencies:", ", ".join(pkgs))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade-strategy", "only-if-needed"] + pkgs, check=True)

import openai
import PIL
print(f"openai={openai.__version__} | pillow={PIL.__version__}")

In [ ]:
# ==============================================================================
# CELL 4: Review & Apply Manual Subtitle Speaker Overrides (If Needed)
# ==============================================================================
import os
import sys

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = "/content/MovieRag_repo" if os.path.isdir("/content/MovieRag_repo/pipeline") else "/content/MovieRag_repo/MovieRag"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from pipeline.stage4_subtitles import apply_subtitle_speaker_fixes

_movie_id = globals().get("MOVIE_ID", "sintel")
_drive_root = globals().get("DRIVE_ROOT", "/content/drive/MyDrive/MovieRAG")
_workdir = globals().get("WORKDIR", f"{_drive_root}/workdir/{_movie_id}")

# Pure manual overrides keyed by utt_id (e.g. "utt_0009"), 2-decimal start_time ("143.22"), or exact text.
# Leave empty {} if you already applied overrides in Notebook 2 Cell 6B, or list any overrides below:
SPEAKER_OVERRIDES = {
    # "utt_0007": "Sintel",
}

fix_summary = apply_subtitle_speaker_fixes(
    output_dir=_workdir,
    speaker_overrides=SPEAKER_OVERRIDES,
)
print(f"Applied {fix_summary['updated_count']} manual override(s) in attributed_subtitles.json & shots_and_scenes.json")
print(f"Speaker distribution: {fix_summary['speaker_counts']}")
print("\n--- SUBTITLES GOING INTO VLM PROMPT ---")
for i, u in enumerate(fix_summary["utterances"]):
    uid = u.get("utt_id", f"utt_{i:04d}")
    print(f"  {uid} [{u['start_time']:6.2f}s - {u['end_time']:6.2f}s] {u['speaker']:8s} ({u['source']}): {u['text']}")

In [ ]:
# ==============================================================================
# CELL 5 (OPTIONAL DIAGNOSTIC): Test 1 Scene with deepseek-flash Before Full Run
# ==============================================================================
import os
import sys

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = "/content/MovieRag_repo" if os.path.isdir("/content/MovieRag_repo/pipeline") else "/content/MovieRag_repo/MovieRag"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from pipeline.stage5_vlm_caption import run_single_scene_diagnostic

_movie_id = globals().get("MOVIE_ID", "sintel")
_movie_title = globals().get("MOVIE_TITLE", "Sintel")
_drive_root = globals().get("DRIVE_ROOT", "/content/drive/MyDrive/MovieRAG")
_workdir = globals().get("WORKDIR", f"{_drive_root}/workdir/{_movie_id}")
_char_bank = globals().get("CHAR_BANK_DIR", f"{_drive_root}/char_bank/{_movie_id}")
_vlm_model = globals().get("VLM_MODEL_ID", "deepseek-flash")
_api_base = globals().get("API_BASE_URL", "https://api.deepseek.com")
_max_kfs = globals().get("MAX_KEYFRAMES_PER_SCENE", 15)
_max_w = globals().get("MAX_IMAGE_WIDTH", 960)

DIAGNOSTIC_SCENE_INDEX = 4  # sintel_scene_0004 (Shaman hut scene)

_diag = run_single_scene_diagnostic(
    output_dir=_workdir,
    scene_index=DIAGNOSTIC_SCENE_INDEX,
    movie_title=_movie_title,
    char_bank_dir=_char_bank,
    vlm_model_id=_vlm_model,
    api_base_url=_api_base,
    max_keyframes_per_scene=_max_kfs,
    max_image_width=_max_w,
)

In [ ]:
# ==============================================================================
# CELL 6: Stage A5 - Full 34-Scene deepseek-flash Captioning & Bundle Assembly
# ==============================================================================
import os
import sys

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = "/content/MovieRag_repo" if os.path.isdir("/content/MovieRag_repo/pipeline") else "/content/MovieRag_repo/MovieRag"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from pipeline.stage5_vlm_caption import run_stage5_vlm_caption

_movie_id = globals().get("MOVIE_ID", "sintel")
_movie_title = globals().get("MOVIE_TITLE", "Sintel")
_franchise_id = globals().get("FRANCHISE_ID", _movie_id)
_drive_root = globals().get("DRIVE_ROOT", "/content/drive/MyDrive/MovieRAG")
_workdir = globals().get("WORKDIR", f"{_drive_root}/workdir/{_movie_id}")
_char_bank = globals().get("CHAR_BANK_DIR", f"{_drive_root}/char_bank/{_movie_id}")
_vlm_model = globals().get("VLM_MODEL_ID", "deepseek-flash")
_api_base = globals().get("API_BASE_URL", "https://api.deepseek.com")
_max_kfs = globals().get("MAX_KEYFRAMES_PER_SCENE", 15)
_max_w = globals().get("MAX_IMAGE_WIDTH", 960)

# Keep RESTART_STAGE5 = True on your first deepseek-flash run so old Qwen outputs are archived
# into workdir/sintel/archive_prev_vlm/ and all 34 scenes are captioned fresh.
RESTART_STAGE5 = True  # @param {type:"boolean"}

stage5_summary = run_stage5_vlm_caption(
    output_dir=_workdir,
    movie_title=_movie_title,
    franchise_id=_franchise_id,
    char_bank_dir=_char_bank,
    vlm_model_id=_vlm_model,
    api_base_url=_api_base,
    max_keyframes_per_scene=_max_kfs,
    max_image_width=_max_w,
    restart=RESTART_STAGE5,
)

print("\nStage A5 Summary:")
for k, v in stage5_summary.items():
    print(f"  {k}: {v}")

In [ ]:
# ==============================================================================
# CELL 7: Verify Final movie_knowledge_bundle.json Schema & Content
# ==============================================================================
import json
from pathlib import Path

_movie_id = globals().get("MOVIE_ID", "sintel")
_workdir = Path(globals().get("WORKDIR", f"/content/drive/MyDrive/MovieRAG/workdir/{_movie_id}"))
bundle_path = _workdir / "movie_knowledge_bundle.json"

with open(bundle_path, "r", encoding="utf-8") as f:
    bundle = json.load(f)

scenes = bundle.get("scenes", [])
print("=== PHASE A FINAL BUNDLE VERIFICATION ===")
print(f"  Bundle File           : {bundle_path} ({bundle_path.stat().st_size / (1024**2):.2f} MB)")
print(f"  Movie ID / Title      : {bundle.get('movie_id')} / {bundle.get('movie_title')}")
print(f"  Duration / FPS        : {bundle.get('duration_sec')}s @ {bundle.get('fps')} fps")
print(f"  Glossary Characters   : {list(bundle.get('character_glossary', {}).keys())}")
print(f"  External Plot Docs    : {len(bundle.get('external_plot_documents', []))} (tiers: {[d.get('tier') for d in bundle.get('external_plot_documents', [])]})")
print(f"  Total Scenes          : {len(scenes)}")
print(f"  Scenes w/ vlm_caption : {sum(1 for s in scenes if s.get('vlm_caption'))} / {len(scenes)}")
print(f"  Scenes w/ exit_state  : {sum(1 for s in scenes if s.get('exit_state'))} / {len(scenes)}")
print(f"  Scenes w/ subtitles   : {sum(1 for s in scenes if s.get('subtitles'))} / {len(scenes)}")

for sc in [scenes[1], scenes[4], scenes[6], scenes[24]] if len(scenes) >= 25 else scenes[:3]:
    print(f"\n--- {sc['scene_id']} [{sc['start_time']:.1f}s - {sc['end_time']:.1f}s] ---")
    print(f"  Characters : {[c['name'] for c in sc.get('characters_present', [])]}")
    print(f"  Subtitles  : {[(u['speaker'], u['text']) for u in sc.get('subtitles', [])]}")
    print(f"  Caption    : {sc.get('vlm_caption', '')[:320]}...")
    print(f"  Exit State : {sc.get('exit_state', '')}")